In [1]:
import config
import requests
import json
import pandas as pd
import time

NYT_KEY = config.NYT_KEY

In [44]:
# create NYT as JSON

sections = ['arts', 'automobiles', 'books/review', 'business', 'fashion', 'food', 'health', 'home', 'insider', 'magazine', 'movies', 'nyregion', 'obituaries', 'opinion', 'politics', 'realestate', 'science', 'sundayreview', 'technology', 'theater', 't-magazine', 'travel', 'upshot', 'us', 'world']
# These are different versions of this variable that were used test different scenarios.
# sections = ['business', 'nyregion', 'travel', 'us']
# sections = ['arts', 'sundayreview','technology', 'theater', 't-magazine', 'travel', 'upshot', 'us', 'world']
# sectionDict = {}
sectionDF = pd.DataFrame()
iterator = 0
for x in sections:
    ENDPOINT = f'https://api.nytimes.com/svc/topstories/v2/{x}.json'
    PARAMETERS = {
                    'api-key': NYT_KEY
                 }
    
    tempURL = requests.get(url = ENDPOINT, params = PARAMETERS)
    tempURL2 = tempURL.json()
    # print(tempURL2) test to see if api loads
    
    # this part skips any sections that may not be in the current top stories; error otherwise
    if 'results' not in tempURL2:
        print(f"Skipping {x} — not a valid Top Stories section \n")
        continue
    
    # converts results to dataframe
    tempDF = pd.DataFrame(tempURL2['results'])
    # removes the word "By " from the 'byline' column
    tempDF['byline'] = tempDF['byline'].str.replace('By ','')
    # makes a copy of the dataframe with just the following "columns": 'section','title','byline','updated_date','created_date'
    byDF = tempDF[['section','title','byline','updated_date','created_date']].copy()
    # adds the current iteration of the dataframe to the end of another dataframe to make one cumulative one
    sectionDF = pd.concat([sectionDF,byDF],ignore_index=True)

    iterator += 1
    
    # # # since there is a limit of 5 calls per 60s, we must input a delay if we want to get all the data from all of the NYT sections
    if iterator % 5 == 0:
        print("Pausing for the API")
        time.sleep(60)
        print("Continuing...")

print("All data has been loaded")


Skipping books/review — not a valid Top Stories section 

Pausing for the API
Continuing...
Pausing for the API
Continuing...
Pausing for the API
Continuing...
Pausing for the API
Continuing...


In [47]:
# display the raw data gathered from the API
sectionDF

,section,title,byline,updated_date,created_date
0,theater,Zora Neale Hurston’s Play Comes Alive for the ...,Salamishah Tillet,2025-10-11T07:23:57-04:00,2025-10-11T05:01:26-04:00
1,arts,"Beheaded and Sent to Watery Graves, Columbus S...",Julia Jacobs,2025-10-11T05:02:00-04:00,2025-10-11T05:02:00-04:00
2,arts,Master of a Thousand Satisfied Soaks,Patricia Leigh Brown,2025-10-11T11:05:34-04:00,2025-10-11T05:00:48-04:00
3,arts,Manga Is a Pop Culture Phenomenon. It’s Also a...,Maya Phillips,2025-10-11T05:01:22-04:00,2025-10-11T05:01:22-04:00
4,arts,Jesse Williams Feels Like He’s Just Getting St...,Kathryn Shattuck,2025-10-11T05:00:36-04:00,2025-10-11T05:00:36-04:00
...,...,...,...,...,...
713,world,Tensions have been rising between Venezuela an...,Jonathan Wolfe,2025-10-10T10:41:28-04:00,2025-10-10T10:41:28-04:00
714,world,Machado’s Peace Prize Is Latest Nobel for Fema...,Hannah Beech,2025-10-10T08:04:16-04:00,2025-10-10T08:04:16-04:00
715,world,France’s Domestic Instability Has Weakened Its...,Steven Erlanger,2025-10-11T02:03:24-04:00,2025-10-10T07:45:29-04:00
716,world,"Who Is María Corina Machado, Winner of the 202...",Jonathan Wolfe and Julie Turkewitz,2025-10-10T17:53:07-04:00,2025-10-10T06:32:21-04:00


In [45]:
# count how many posts each author has made
count = sectionDF[['byline']].copy()
count = pd.DataFrame(count.value_counts()).reset_index()
count

,byline,count
0,,26
1,Sarah Bahr,16
2,Jack Ewing,13
3,Vanessa Friedman,13
4,Nate Cohn,7
...,...,...
422,Andrea Bussell,1
423,Andrew Duehren,1
424,"Andrew Ross Sorkin, Bernhard Warner, Sarah Kes...",1
425,Andrew Zimmern,1


In [46]:
# check what posts were modified after they were originally uploaded
articleUpdated = pd.DataFrame(sectionDF[['title','byline','section','updated_date','created_date']].copy())
boolUpdated = articleUpdated.copy()

boolUpdated['modified_after_post'] = articleUpdated['updated_date'] != articleUpdated['created_date']

modCounter = boolUpdated['modified_after_post'].value_counts()

print("Count of articles posted today that were modified after they were initially posted.")
print(modCounter)

boolUpdated

Count of articles posted today that were modified after they were initially posted.
modified_after_post
True     627
False     91
Name: count, dtype: int64


,title,byline,section,updated_date,created_date,modified_after_post
0,Zora Neale Hurston’s Play Comes Alive for the ...,Salamishah Tillet,theater,2025-10-11T07:23:57-04:00,2025-10-11T05:01:26-04:00,True
1,"Beheaded and Sent to Watery Graves, Columbus S...",Julia Jacobs,arts,2025-10-11T05:02:00-04:00,2025-10-11T05:02:00-04:00,False
2,Master of a Thousand Satisfied Soaks,Patricia Leigh Brown,arts,2025-10-11T11:05:34-04:00,2025-10-11T05:00:48-04:00,True
3,Manga Is a Pop Culture Phenomenon. It’s Also a...,Maya Phillips,arts,2025-10-11T05:01:22-04:00,2025-10-11T05:01:22-04:00,False
4,Jesse Williams Feels Like He’s Just Getting St...,Kathryn Shattuck,arts,2025-10-11T05:00:36-04:00,2025-10-11T05:00:36-04:00,False
...,...,...,...,...,...,...
713,Tensions have been rising between Venezuela an...,Jonathan Wolfe,world,2025-10-10T10:41:28-04:00,2025-10-10T10:41:28-04:00,False
714,Machado’s Peace Prize Is Latest Nobel for Fema...,Hannah Beech,world,2025-10-10T08:04:16-04:00,2025-10-10T08:04:16-04:00,False
715,France’s Domestic Instability Has Weakened Its...,Steven Erlanger,world,2025-10-11T02:03:24-04:00,2025-10-10T07:45:29-04:00,True
716,"Who Is María Corina Machado, Winner of the 202...",Jonathan Wolfe and Julie Turkewitz,world,2025-10-10T17:53:07-04:00,2025-10-10T06:32:21-04:00,True
